In [ ]:
!pip install -q gradio pandas numpy scikit-learn sentence-transformers faiss-cpu google-genai

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("MY_API"))

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [ ]:
import pandas as pd

data = {
    "Complaint_ID": ["G001", "G002", "G003", "G004", "G005", "G006"],
    "Department": ["Water", "Electricity", "Transport", "Water", "Healthcare", "Transport"],
    "Location": ["Coimbatore", "Chennai", "Salem", "Madurai", "Erode", "Coimbatore"],
    "Complaint": [
        "Irregular water supply for several days",
        "Frequent power cuts in residential areas",
        "Damaged roads causing accidents",
        "Water leakage remains unresolved",
        "Shortage of medicines in government hospital",
        "Buses frequently arrive late"
    ],
    "Status": ["Pending", "Resolved", "Pending", "Reopened", "Pending", "Resolved"]
}

df = pd.DataFrame(data)
print(df)

  Complaint_ID   Department    Location  \
0         G001        Water  Coimbatore   
1         G002  Electricity     Chennai   
2         G003    Transport       Salem   
3         G004        Water     Madurai   
4         G005   Healthcare       Erode   
5         G006    Transport  Coimbatore   

                                      Complaint    Status  
0       Irregular water supply for several days   Pending  
1      Frequent power cuts in residential areas  Resolved  
2               Damaged roads causing accidents   Pending  
3              Water leakage remains unresolved  Reopened  
4  Shortage of medicines in government hospital   Pending  
5                  Buses frequently arrive late  Resolved  


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["Complaint"].tolist())
index = faiss.IndexFlatIP(embeddings.shape[1])

faiss.normalize_L2(embeddings)
index.add(embeddings.astype("float32"))

print("RAG knowledge index created!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG knowledge index created!


In [ ]:
def retrieve_complaints(question):
    query = model.encode([question]).astype("float32")
    faiss.normalize_L2(query)

    scores, ids = index.search(query, 3)
    return df.iloc[ids[0]]

print(retrieve_complaints("Why is water supply irregular?"))

  Complaint_ID   Department    Location  \
0         G001        Water  Coimbatore   
3         G004        Water     Madurai   
1         G002  Electricity     Chennai   

                                  Complaint    Status  
0   Irregular water supply for several days   Pending  
3          Water leakage remains unresolved  Reopened  
1  Frequent power cuts in residential areas  Resolved  


In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("MY_API").strip()
)

print("API client refreshed!")

API client refreshed!


In [9]:
!pip install -q openai

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("MY_API"),
    base_url="https://nexusapi.navigatelabs.ai"
)

models = client.models.list()
print([m.id for m in models.data])

['nano-banana', 'whisper-1', 'nova-micro', 'text-embedding-3-small', 'gpt-4o-mini-tts', 'gpt-4.1-nano']


In [10]:
response = client.chat.completions.create(
    model="nova-micro",
    messages=[
        {"role": "user", "content": "Reply: AI is working!"}
    ]
)

print(response.choices[0].message.content)

It sounds like your AI system is up and running successfully! If you need any assistance or have any specific questions about your AI implementation, feel free to ask. Whether it's troubleshooting, optimizing performance, or exploring new features, I'm here to help. How can I assist you further with your AI project?


In [11]:
def grievance_rag(question):
    evidence = retrieve_complaints(question).to_string(index=False)

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {"role": "system", "content": "Answer only using the supplied complaint records. Never invent facts. Mention complaint IDs."},
            {"role": "user", "content": f"Evidence:\n{evidence}\n\nQuestion: {question}"}
        ]
    )

    return response.choices[0].message.content

print(grievance_rag("Why are residents facing water supply problems?"))

Based on the provided complaint records, residents are facing water supply problems due to the irregular water supply for several days in Coimbatore (Complaint ID: G001). The complaint is currently in the "Pending" status.


In [12]:
pattern_table = pd.crosstab(
    df["Department"], df["Status"]
)

pattern_table["Total"] = pattern_table.sum(axis=1)

print(pattern_table)

Status       Pending  Reopened  Resolved  Total
Department                                     
Electricity        0         0         1      1
Healthcare         1         0         0      1
Transport          1         0         1      2
Water              1         1         0      2


In [13]:
response = client.chat.completions.create(
    model="nova-micro",
    messages=[
        {"role": "user", "content": f"""
        Analyse these public grievance statistics.
        Identify departments with the most unresolved complaints.
        Suggest practical improvements.

        {pattern_table.to_string()}
        """}
    ]
)

pattern_insights = response.choices[0].message.content
print(pattern_insights)

To analyze the public grievance statistics and identify the departments with the most unresolved complaints, we can break down the data and then suggest practical improvements for the departments that are struggling the most.

### Analysis

1. **Pending Complaints:**
   - Electricity: 0
   - Healthcare: 1
   - Transport: 1
   - Water: 1

2. **Reopened Complaints:**
   - Electricity: 0
   - Healthcare: 0
   - Transport: 0
   - Water: 1

3. **Resolved Complaints:**
   - Electricity: 1
   - Healthcare: 0
   - Transport: 1
   - Water: 0

4. **Total Complaints:**
   - Electricity: 1
   - Healthcare: 1
   - Transport: 2
   - Water: 2

From the statistics, it is evident that the departments with the most unresolved complaints are:

1. **Healthcare:**
   - Pending: 1
   - Reopened: 0
   - Resolved: 0
   - Total: 1

2. **Water:**
   - Pending: 1
   - Reopened: 1
   - Resolved: 0
   - Total: 2

3. **Transport:**
   - Pending: 1
   - Reopened: 0
   - Resolved: 1
   - Total: 2

### Practical Impro

In [14]:
def prioritisation_agent():
    results = []

    for dept, group in df.groupby("Department"):
        unresolved = group[group["Status"].isin(
            ["Pending", "In Progress", "Reopened"]
        )].shape[0]

        priority = "High" if unresolved >= 2 else "Low"

        results.append({
            "Department": dept,
            "Unresolved": unresolved,
            "Priority": priority
        })

    return pd.DataFrame(results)

priority_table = prioritisation_agent()
print(priority_table)

    Department  Unresolved Priority
0  Electricity           0      Low
1   Healthcare           1      Low
2    Transport           1      Low
3        Water           2     High


In [15]:
def investigation_agent(department):
    result = df[df["Department"].str.lower() == department.lower()]

    if result.empty:
        return "No matching complaints found."

    return result[[
        "Complaint_ID", "Location", "Complaint", "Status"
    ]]

print(investigation_agent("Water"))

  Complaint_ID    Location                                Complaint    Status
0         G001  Coimbatore  Irregular water supply for several days   Pending
3         G004     Madurai         Water leakage remains unresolved  Reopened


In [16]:
import gradio as gr

def dashboard(question):
    return grievance_agent(question)

demo = gr.Interface(
    fn=dashboard,
    inputs=gr.Textbox(
        label="Enter your grievance question",
        placeholder="Why are residents facing water supply problems?"
    ),
    outputs=gr.Textbox(label="AI Investigation Report", lines=12),
    title="AI-Driven Public Grievance Intelligence System",
    description="Investigate public complaints using AI, RAG and multi-agent analysis."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://400c0f70ce4fc15a84.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [40]:

import gradio as gr
import plotly.express as px

gr.close_all()

# Refresh dashboard information after dataset changes
def build_dashboard_outputs(message):

    global pattern_table, priority_table, pattern_insights

    pattern_table = pd.crosstab(
        df["Department"], df["Status"]
    )

    priority_table = prioritisation_agent()

    # KPI values
    total = len(df)
    pending = (df["Status"] == "Pending").sum()
    resolved = (df["Status"] == "Resolved").sum()
    reopened = (df["Status"] == "Reopened").sum()
    in_progress = (df["Status"] == "In Progress").sum()

    # Charts
    status_chart = px.pie(
        df, names="Status",
        title="Complaint Status Distribution"
    )

    department_counts = (
        df["Department"].value_counts()
        .reset_index()
    )
    department_counts.columns = ["Department", "Complaints"]

    department_chart = px.bar(
        department_counts,
        x="Department",
        y="Complaints",
        title="Department-wise Complaints"
    )

    # Refresh AI pattern insights
    try:
        response = client.chat.completions.create(
            model="nova-micro",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Analyse only the supplied complaint statistics. "
                        "Identify patterns and suggest actions. "
                        "Do not invent facts."
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Complaint statistics:\n"
                        + pattern_table.to_string()
                    )
                }
            ]
        )
        pattern_insights = response.choices[0].message.content

    except Exception as e:
        pattern_insights = "AI insight refresh failed: " + str(e)

    return (
        message,
        df.head(20),
        df,
        pattern_table,
        priority_table,
        total,
        pending,
        resolved,
        reopened,
        in_progress,
        status_chart,
        department_chart,
        pattern_insights
    )


def upload_and_refresh(file):

    message = activate_uploaded_dataset(file)

    return build_dashboard_outputs(message)


def restore_and_refresh():

    message = reset_to_default_dataset()

    return build_dashboard_outputs(message)


def filter_active_dataset(department, location, status):

    result = df.copy()

    if department != "All":
        result = result[result["Department"] == department]

    if location != "All":
        result = result[result["Location"] == location]

    if status != "All":
        result = result[result["Status"] == status]

    return result


# Initial dashboard charts
status_chart = px.pie(
    df, names="Status",
    title="Complaint Status Distribution"
)

department_counts = df["Department"].value_counts().reset_index()
department_counts.columns = ["Department", "Complaints"]

department_chart = px.bar(
    department_counts,
    x="Department",
    y="Complaints",
    title="Department-wise Complaints"
)

# Dashboard
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue"
    ),
    title="Public Grievance Intelligence"
) as demo:

    gr.Markdown("""
    # 🏛️ AI-Driven Public Grievance Intelligence
    ### Smart monitoring • Pattern discovery • AI investigation
    """)

    with gr.Tab("📊 Overview"):

        with gr.Row():
            total_box = gr.Number(value=len(df), label="Total Complaints")
            pending_box = gr.Number(
                value=(df["Status"] == "Pending").sum(),
                label="Pending"
            )
            resolved_box = gr.Number(
                value=(df["Status"] == "Resolved").sum(),
                label="Resolved"
            )
            reopened_box = gr.Number(
                value=(df["Status"] == "Reopened").sum(),
                label="Reopened"
            )
            progress_box = gr.Number(
                value=(df["Status"] == "In Progress").sum(),
                label="In Progress"
            )

        with gr.Row():
            status_plot = gr.Plot(value=status_chart)
            department_plot = gr.Plot(value=department_chart)

    with gr.Tab("🔎 Complaint Explorer"):

        with gr.Row():
            dept = gr.Dropdown(
                ["All"] + sorted(df["Department"].unique().tolist()),
                value="All", label="Department"
            )

            loc = gr.Dropdown(
                ["All"] + sorted(df["Location"].unique().tolist()),
                value="All", label="Location"
            )

            stat = gr.Dropdown(
                ["All"] + sorted(df["Status"].unique().tolist()),
                value="All", label="Status"
            )

        complaint_table = gr.Dataframe(value=df)

        for control in [dept, loc, stat]:
            control.change(
                filter_active_dataset,
                [dept, loc, stat],
                complaint_table
            )

    with gr.Tab("🧠 Pattern Discovery"):

        gr.Markdown("## AI-Discovered Complaint Patterns")

        insights_box = gr.Markdown(value=pattern_insights)

        gr.Markdown("### Department-wise Complaint Statistics")

        pattern_box = gr.Dataframe(value=pattern_table)

    with gr.Tab("⚠️ Priority Analysis"):

        gr.Markdown("## Department Priority Assessment")

        priority_box = gr.Dataframe(value=priority_table)

    with gr.Tab("🤖 AI Investigation"):

        gr.Markdown("## Ask the Grievance Intelligence Agent")

        question = gr.Textbox(
            label="Enter your question",
            placeholder="Which grievances remain unresolved in Coimbatore?"
        )

        investigate = gr.Button(
            "🔍 Investigate",
            variant="primary"
        )

        answer = gr.Markdown()

        investigate.click(
            grievance_agent,
            inputs=question,
            outputs=answer
        )

    with gr.Tab("📂 Dataset Management"):

        gr.Markdown("""
        ## Manage Complaint Dataset

        Upload your own complaint dataset or return to the
        original default dataset.

        **Supported formats:** CSV and Excel.
        """)

        upload_file = gr.File(
            label="Upload Complaint Dataset",
            file_types=[".csv", ".xlsx", ".xls"],
            type="filepath"
        )

        upload_button = gr.Button(
            "📤 Upload and Analyse",
            variant="primary"
        )

        restore_button = gr.Button(
            "↩️ Restore Default Dataset"
        )

        upload_status = gr.Markdown()

        gr.Markdown("### Active Dataset Preview")

        dataset_preview = gr.Dataframe(value=df.head(20))

        # Components refreshed after upload or reset
        refresh_outputs = [
            upload_status,
            dataset_preview,
            complaint_table,
            pattern_box,
            priority_box,
            total_box,
            pending_box,
            resolved_box,
            reopened_box,
            progress_box,
            status_plot,
            department_plot,
            insights_box
        ]

        upload_button.click(
            upload_and_refresh,
            inputs=upload_file,
            outputs=refresh_outputs
        )

        restore_button.click(
            restore_and_refresh,
            inputs=[],
            outputs=refresh_outputs
        )

    gr.Markdown("""
    ---
    **Default Dataset:** 1,002 simulated complaint records.
    This is an academic prototype, not an official government database.
    """)

demo.launch(share=True)


Closing server running on port: 7860


/tmp/ipykernel_10173/4262666668.py:134: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a19a05353bb167fd3b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
def grievance_agent(question):
    location = next(
        (x for x in df["Location"].unique()
         if x.lower() in question.lower()), None
    )

    if location:
        evidence = df[
            (df["Location"] == location) &
            (df["Status"].isin(["Pending", "In Progress", "Reopened"]))
        ]
    else:
        evidence = retrieve_complaints(question)

    prompt = f"""
    Answer using only these complaint records.
    Mention complaint IDs and statuses.
    If no records are found, say so.
    Question: {question}
    Evidence:
    {evidence.to_string(index=False)}
    """

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [21]:
!pip install -q plotly

In [22]:
import plotly.express as px

total = len(df)
pending = (df["Status"] == "Pending").sum()
resolved = (df["Status"] == "Resolved").sum()
reopened = (df["Status"] == "Reopened").sum()

print("Dashboard statistics ready!")
print("Total:", total)
print("Pending:", pending)
print("Resolved:", resolved)
print("Reopened:", reopened)

Dashboard statistics ready!
Total: 6
Pending: 3
Resolved: 2
Reopened: 1


In [23]:
status_chart = px.pie(
    df,
    names="Status",
    title="Complaint Status Distribution",
    hole=0.45
)

department_chart = px.histogram(
    df,
    x="Department",
    color="Status",
    title="Department-wise Complaints",
    barmode="group"
)

status_chart.show()
department_chart.show()

In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output

dept_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Department"].unique().tolist()),
    description="Department:"
)

location_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Location"].unique().tolist()),
    description="Location:"
)

status_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Status"].unique().tolist()),
    description="Status:"
)

def filter_complaints(change=None):
    result = df.copy()

    if dept_filter.value != "All":
        result = result[result["Department"] == dept_filter.value]

    if location_filter.value != "All":
        result = result[result["Location"] == location_filter.value]

    if status_filter.value != "All":
        result = result[result["Status"] == status_filter.value]

    clear_output(wait=True)
    display(dept_filter, location_filter, status_filter)
    display(result)

for control in [dept_filter, location_filter, status_filter]:
    control.observe(filter_complaints, names="value")

filter_complaints()

Dropdown(description='Department:', options=('All', 'Electricity', 'Healthcare', 'Transport', 'Water'), value=…

Dropdown(description='Location:', options=('All', 'Chennai', 'Coimbatore', 'Erode', 'Madurai', 'Salem'), value…

Dropdown(description='Status:', options=('All', 'Pending', 'Reopened', 'Resolved'), value='All')

,Complaint_ID,Department,Location,Complaint,Status
0,G001,Water,Coimbatore,Irregular water supply for several days,Pending
1,G002,Electricity,Chennai,Frequent power cuts in residential areas,Resolved
2,G003,Transport,Salem,Damaged roads causing accidents,Pending
3,G004,Water,Madurai,Water leakage remains unresolved,Reopened
4,G005,Healthcare,Erode,Shortage of medicines in government hospital,Pending
5,G006,Transport,Coimbatore,Buses frequently arrive late,Resolved


In [25]:
import gradio as gr

def filter_dashboard(department, location, status):
    result = df.copy()

    if department != "All":
        result = result[result["Department"] == department]

    if location != "All":
        result = result[result["Location"] == location]

    if status != "All":
        result = result[result["Status"] == status]

    return result

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"),
    title="Public Grievance Intelligence"
) as demo:

    gr.Markdown("""
    # 🏛️ Public Grievance Intelligence
    ### AI-powered complaint analysis and accountability dashboard

    Monitor public complaints, discover patterns and investigate unresolved issues.
    """)

    with gr.Tab("📊 Overview"):
        gr.Markdown("### Complaint Statistics")

        with gr.Row():
            gr.Number(value=total, label="Total Complaints")
            gr.Number(value=pending, label="Pending")
            gr.Number(value=resolved, label="Resolved")
            gr.Number(value=reopened, label="Reopened")

        with gr.Row():
            gr.Plot(value=status_chart)
            gr.Plot(value=department_chart)

    with gr.Tab("🔎 Complaint Explorer"):
        gr.Markdown("### Search and filter complaints")

        with gr.Row():
            dept = gr.Dropdown(
                ["All"] + sorted(df["Department"].unique().tolist()),
                value="All", label="Department"
            )
            loc = gr.Dropdown(
                ["All"] + sorted(df["Location"].unique().tolist()),
                value="All", label="Location"
            )
            stat = gr.Dropdown(
                ["All"] + sorted(df["Status"].unique().tolist()),
                value="All", label="Status"
            )

        complaint_table = gr.Dataframe(value=df, interactive=False)

        for control in [dept, loc, stat]:
            control.change(
                filter_dashboard,
                inputs=[dept, loc, stat],
                outputs=complaint_table
            )

    with gr.Tab("⚠️ Priority Analysis"):
        gr.Markdown("### Department-wise Priority")
        gr.Dataframe(value=priority_table, interactive=False)

    with gr.Tab("🤖 AI Investigation"):
        gr.Markdown("### Ask the Grievance Intelligence Agent")

        question = gr.Textbox(
            label="Your question",
            placeholder="Which grievances remain unresolved in Coimbatore?"
        )

        investigate = gr.Button("🔍 Investigate", variant="primary")
        answer = gr.Markdown(label="Investigation Report")

        investigate.click(
            grievance_agent,
            inputs=question,
            outputs=answer
        )

    gr.Markdown("""
    ---
    **Prototype notice:** This demonstration uses six simulated complaint records.
    It does not represent actual government grievance data.
    """)

gr.close_all()
demo.launch(share=True)

/tmp/ipykernel_10173/1284870444.py:17: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Closing server running on port: 7860
Closing server running on port: 7861
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00684c643d422e234f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

complaints = {
    "Water": [
        "Irregular water supply in residential areas",
        "Drinking water contamination reported",
        "Water pipeline leakage remains unresolved",
        "Low water pressure in several streets",
        "Water tanker delivery is frequently delayed"
    ],
    "Electricity": [
        "Frequent power cuts in residential areas",
        "Damaged electric poles require repair",
        "Streetlights are not functioning",
        "Electricity voltage fluctuations reported",
        "Delayed resolution of power complaints"
    ],
    "Transport": [
        "Buses frequently arrive late",
        "Insufficient buses on busy routes",
        "Damaged roads causing traffic problems",
        "Bus stops lack proper facilities",
        "Poor maintenance of public transport"
    ],
    "Healthcare": [
        "Shortage of medicines in government hospital",
        "Long waiting time at government hospitals",
        "Insufficient medical staff at health centres",
        "Poor sanitation in hospital premises",
        "Delayed medical services for patients"
    ],
    "Sanitation": [
        "Garbage is not collected regularly",
        "Overflowing public dustbins",
        "Blocked drainage causing water stagnation",
        "Unclean public spaces",
        "Irregular waste disposal services"
    ],
    "Roads": [
        "Potholes on major roads",
        "Damaged roads require immediate repair",
        "Poor road maintenance in residential areas",
        "Road construction work remains incomplete",
        "Missing road safety signs"
    ],
    "Education": [
        "Insufficient classroom facilities",
        "Shortage of teachers in government schools",
        "Poor maintenance of school buildings",
        "Lack of drinking water in schools",
        "Inadequate sanitation facilities in schools"
    ],
    "Public Safety": [
        "Non-functional streetlights create safety concerns",
        "Damaged public safety barriers",
        "Unsafe pedestrian crossings",
        "Lack of traffic signals at busy junctions",
        "Delayed response to public safety complaints"
    ],
    "Revenue": [
        "Delay in processing government certificates",
        "Property tax payment records contain errors",
        "Long waiting time at public service offices",
        "Delay in issuing income certificates",
        "Pending applications at revenue offices"
    ],
    "Housing": [
        "Delay in processing housing scheme applications",
        "Poor maintenance of public housing",
        "Water leakage in government housing units",
        "Incomplete housing repair work",
        "Delay in housing benefit approval"
    ]
}

locations = [
    "Coimbatore", "Chennai", "Madurai", "Salem", "Erode",
    "Tiruppur", "Trichy", "Tirunelveli", "Vellore", "Thanjavur",
    "Dindigul", "Namakkal", "Karur", "Kanchipuram", "Theni",
    "Thoothukudi", "Krishnagiri", "Cuddalore", "Sivaganga", "Nilgiris"
]

statuses = ["Pending", "Resolved", "Reopened", "In Progress"]
status_weights = [0.35, 0.40, 0.10, 0.15]

records = []

for i in range(1000):
    department = random.choice(list(complaints.keys()))

    records.append({
        "Complaint_ID": f"GRV{i+1:04d}",
        "Department": department,
        "Location": random.choice(locations),
        "Complaint": random.choice(complaints[department]),
        "Status": random.choices(
            statuses, weights=status_weights, k=1
        )[0],
        "Complaint_Date": pd.Timestamp("2025-01-01") +
            pd.Timedelta(days=random.randint(0, 320))
    })

df = pd.DataFrame(records)

# Preserve the original demonstration cases
df = pd.concat([
    pd.DataFrame([{
        "Complaint_ID": "G001",
        "Department": "Water",
        "Location": "Coimbatore",
        "Complaint": "Irregular water supply for several days",
        "Status": "Pending",
        "Complaint_Date": pd.Timestamp("2025-11-17")
    }, {
        "Complaint_ID": "G006",
        "Department": "Transport",
        "Location": "Coimbatore",
        "Complaint": "Buses frequently arrive late",
        "Status": "Resolved",
        "Complaint_Date": pd.Timestamp("2025-11-16")
    }]),
    df
], ignore_index=True)

print("Dataset created successfully!")
print("Total complaints:", len(df))
display(df.head())

Dataset created successfully!
Total complaints: 1002


,Complaint_ID,Department,Location,Complaint,Status,Complaint_Date
0,G001,Water,Coimbatore,Irregular water supply for several days,Pending,2025-11-17
1,G006,Transport,Coimbatore,Buses frequently arrive late,Resolved,2025-11-16
2,GRV0001,Electricity,Coimbatore,Streetlights are not functioning,Pending,2025-03-13
3,GRV0002,Electricity,Cuddalore,Frequent power cuts in residential areas,Resolved,2025-01-17
4,GRV0003,Water,Madurai,Drinking water contamination reported,Pending,2025-11-05


In [27]:
embeddings = model.encode(
    df["Complaint"].tolist(),
    show_progress_bar=True
)

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))

print("RAG index updated successfully!")
print("Indexed complaints:", index.ntotal)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

RAG index updated successfully!
Indexed complaints: 1002


In [28]:
total = len(df)
pending = (df["Status"] == "Pending").sum()
resolved = (df["Status"] == "Resolved").sum()
reopened = (df["Status"] == "Reopened").sum()
in_progress = (df["Status"] == "In Progress").sum()

print("Total complaints:", total)
print("Pending:", pending)
print("Resolved:", resolved)
print("Reopened:", reopened)
print("In Progress:", in_progress)

Total complaints: 1002
Pending: 345
Resolved: 406
Reopened: 94
In Progress: 157


In [29]:
status_chart = px.pie(
    df,
    names="Status",
    title="Complaint Status Distribution",
    hole=0.5
)

department_chart = px.histogram(
    df,
    x="Department",
    color="Status",
    title="Department-wise Complaint Analysis",
    barmode="group"
)

status_chart.show()
department_chart.show()

In [30]:
priority_table = prioritisation_agent()

display(priority_table)

,Department,Unresolved,Priority
0,Education,53,High
1,Electricity,55,High
2,Healthcare,61,High
3,Housing,65,High
4,Public Safety,73,High
5,Revenue,63,High
6,Roads,59,High
7,Sanitation,68,High
8,Transport,47,High
9,Water,52,High


In [31]:
pattern_table = pd.crosstab(
    df["Department"],
    df["Status"]
)

pattern_summary = pattern_table.to_string()

response = client.chat.completions.create(
    model="nova-micro",
    messages=[
        {
            "role": "system",
            "content": "Analyse the supplied simulated public grievance statistics. Identify departments with higher unresolved counts and recurring patterns. Do not invent facts."
        },
        {
            "role": "user",
            "content": f"Complaint statistics:\n{pattern_summary}\n\nIdentify important patterns and suggest actions."
        }
    ]
)

pattern_insights = response.choices[0].message.content

print(pattern_insights)

Based on the provided simulated public grievance statistics, here's an analysis of the unresolved complaints (i.e., those that are "In Progress," "Pending," "Reopened") and recurring patterns across the various departments:

### Unresolved Complaints Breakdown:

1. **Highest Unresolved Counts:**
   - **Public Safety:** 71 unresolved complaints (22 In Progress, 39 Pending, 12 Reopened)
   - **Housing:** 65 unresolved complaints (19 In Progress, 38 Pending, 8 Reopened)
   - **Electricity:** 55 unresolved complaints (12 In Progress, 35 Pending, 8 Reopened)
   - **Sanitation:** 58 unresolved complaints (20 In Progress, 39 Pending, 9 Reopened)
   
2. **Lowest Unresolved Counts:**
   - **Transport:** 27 unresolved complaints (8 In Progress, 27 Pending, 12 Reopened)
   - **Water:** 32 unresolved complaints (10 In Progress, 32 Pending, 10 Reopened)
   - **Roads:** 34 unresolved complaints (14 In Progress, 34 Pending, 11 Reopened)

### Recurring Patterns:

1. **High Pending Counts:**
   - **Ele

In [32]:
gr.close_all()

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue"
    ),
    title="Public Grievance Intelligence"
) as demo:

    gr.Markdown("""
    # 🏛️ AI-Driven Public Grievance Intelligence
    ### Smart monitoring • Pattern discovery • AI investigation
    """)

    with gr.Tab("📊 Overview"):
        with gr.Row():
            gr.Number(value=total, label="Total Complaints")
            gr.Number(value=pending, label="Pending")
            gr.Number(value=resolved, label="Resolved")
            gr.Number(value=reopened, label="Reopened")
            gr.Number(value=in_progress, label="In Progress")

        with gr.Row():
            gr.Plot(value=status_chart)
            gr.Plot(value=department_chart)

    with gr.Tab("🔎 Complaint Explorer"):
        with gr.Row():
            dept = gr.Dropdown(
                ["All"] + sorted(df["Department"].unique().tolist()),
                value="All", label="Department"
            )
            loc = gr.Dropdown(
                ["All"] + sorted(df["Location"].unique().tolist()),
                value="All", label="Location"
            )
            stat = gr.Dropdown(
                ["All"] + sorted(df["Status"].unique().tolist()),
                value="All", label="Status"
            )

        complaint_table = gr.Dataframe(value=df)

        for control in [dept, loc, stat]:
            control.change(
                filter_dashboard,
                [dept, loc, stat],
                complaint_table
            )

    with gr.Tab("🧠 Pattern Discovery"):
        gr.Markdown("## AI-Discovered Complaint Patterns")
        gr.Markdown(pattern_insights)

        gr.Markdown("### Department-wise Complaint Statistics")
        gr.Dataframe(value=pattern_table)

    with gr.Tab("⚠️ Priority Analysis"):
        gr.Markdown("## Department Priority Assessment")
        gr.Dataframe(value=priority_table)

    with gr.Tab("🤖 AI Investigation"):
        gr.Markdown("## Ask the Grievance Intelligence Agent")

        question = gr.Textbox(
            label="Enter your question",
            placeholder="Which grievances remain unresolved in Coimbatore?"
        )

        investigate = gr.Button(
            "🔍 Investigate",
            variant="primary"
        )

        answer = gr.Markdown()

        investigate.click(
            grievance_agent,
            inputs=question,
            outputs=answer
        )

    gr.Markdown("""
    ---
    **Dataset:** 1,002 simulated public grievance records.
    This is an academic prototype, not an official government database.
    """)

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_10173/2683747232.py:3: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ddb970f786bd03dc06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
!pip install -q openpyxl pypdf python-docx

In [34]:
import pandas as pd
from pypdf import PdfReader
from docx import Document
import os

def read_uploaded_file(file):
    filename = file.name
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".csv":
        return pd.read_csv(filename)

    elif extension in [".xlsx", ".xls"]:
        return pd.read_excel(filename)

    elif extension == ".pdf":
        reader = PdfReader(filename)
        text = "\n".join(
            page.extract_text() or "" for page in reader.pages
        )
        return text

    elif extension == ".docx":
        document = Document(filename)
        return "\n".join(
            paragraph.text for paragraph in document.paragraphs
        )

    else:
        return "Unsupported file format."

In [35]:
def prepare_uploaded_dataset(uploaded_df):

    required_columns = [
        "Complaint_ID",
        "Department",
        "Location",
        "Complaint",
        "Status"
    ]

    # Standardise column names
    uploaded_df.columns = [
        col.strip().lower().replace(" ", "_")
        for col in uploaded_df.columns
    ]

    column_mapping = {
        "complaint_id": "Complaint_ID",
        "department": "Department",
        "location": "Location",
        "complaint": "Complaint",
        "status": "Status",
        "complaint_date": "Complaint_Date"
    }

    uploaded_df = uploaded_df.rename(columns=column_mapping)

    missing = [
        col for col in required_columns
        if col not in uploaded_df.columns
    ]

    if missing:
        return None, f"Missing required columns: {missing}"

    if "Complaint_Date" not in uploaded_df.columns:
        uploaded_df["Complaint_Date"] = ""

    uploaded_df = uploaded_df[[
        "Complaint_ID",
        "Department",
        "Location",
        "Complaint",
        "Status",
        "Complaint_Date"
    ]].copy()

    uploaded_df = uploaded_df.dropna(
        subset=["Complaint", "Department", "Location"]
    )

    return uploaded_df, "Dataset uploaded and validated successfully!"

In [36]:
# Preserve our original dataset
if "default_df" not in globals():
    default_df = df.copy()

def activate_uploaded_dataset(file):

    global df, embeddings, index
    global pattern_table, priority_table

    if file is None:
        return "Please upload a file."

    uploaded_data = read_uploaded_file(file)

    if not isinstance(uploaded_data, pd.DataFrame):
        return "Please upload a CSV or Excel complaint dataset."

    prepared_df, message = prepare_uploaded_dataset(uploaded_data)

    if prepared_df is None:
        return message

    if prepared_df.empty:
        return "The uploaded dataset contains no valid complaint records."

    df = prepared_df.reset_index(drop=True)

    # Generate new embeddings
    embeddings = model.encode(
        df["Complaint"].astype(str).tolist(),
        show_progress_bar=True
    )

    faiss.normalize_L2(embeddings)

    # Rebuild FAISS index
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype("float32"))

    # Refresh analytical tables
    pattern_table = pd.crosstab(df["Department"], df["Status"])
    priority_table = prioritisation_agent()

    return f"{message}\nActive records: {len(df)}"

In [37]:
def reset_to_default_dataset():

    global df, embeddings, index
    global pattern_table, priority_table

    df = default_df.copy().reset_index(drop=True)

    embeddings = model.encode(
        df["Complaint"].astype(str).tolist(),
        show_progress_bar=True
    )

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype("float32"))

    pattern_table = pd.crosstab(
        df["Department"],
        df["Status"]
    )

    priority_table = prioritisation_agent()

    return "Default dataset restored successfully! Total records: " + str(len(df))

In [38]:
def read_uploaded_file(file):

    filename = file if isinstance(file, str) else file.name
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".csv":
        return pd.read_csv(filename)

    elif extension in [".xlsx", ".xls"]:
        return pd.read_excel(filename)

    elif extension == ".pdf":
        reader = PdfReader(filename)
        return "\n".join(
            page.extract_text() or "" for page in reader.pages
        )

    elif extension == ".docx":
        document = Document(filename)
        return "\n".join(
            paragraph.text for paragraph in document.paragraphs
        )

    return "Unsupported file format."

In [39]:
def upload_dataset_for_dashboard(file):
    message = activate_uploaded_dataset(file)

    return (
        message,
        df,
        pattern_table,
        priority_table
    )


def restore_default_for_dashboard():
    message = reset_to_default_dataset()

    return (
        message,
        df,
        pattern_table,
        priority_table
    )

In [41]:
sample_data = pd.DataFrame({
    "Complaint_ID": ["TEST001", "TEST002", "TEST003", "TEST004"],
    "Department": ["Water", "Transport", "Electricity", "Sanitation"],
    "Location": ["Coimbatore", "Chennai", "Madurai", "Salem"],
    "Complaint": [
        "Irregular water supply in residential areas",
        "Buses are frequently delayed",
        "Frequent power interruptions",
        "Garbage is not collected regularly"
    ],
    "Status": ["Pending", "Resolved", "In Progress", "Reopened"],
    "Complaint_Date": [
        "2026-11-15", "2026-11-16",
        "2026-11-17", "2026-11-18"
    ]
})

sample_data.to_csv(
    "/content/sample_complaints.csv",
    index=False
)

from google.colab import files
files.download("/content/sample_complaints.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>